# Random Forest for Ransomware Detection

This section performs feature extraction and trains and evaluates the random forest model. Many columns are grouped based on type, which makes analysis easier and reduces dimensionality. 

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Define file paths for data

In [4]:
LOCAL_PATH = "/Users/doro/Library/CloudStorage/OneDrive-Personal/SCHOOL/SJSU/Semester 4/CMPE 209/project/cmpe209-ransomware/"
INPUT_PATH = LOCAL_PATH + "data/processed"
X_TRAIN = pd.read_csv(INPUT_PATH + "/X_train.csv")
Y_TRAIN = pd.read_csv(INPUT_PATH + "/y_train.csv")

X_TEST = pd.read_csv(INPUT_PATH + "/X_test.csv")
Y_TEST = pd.read_csv(INPUT_PATH + "/y_test.csv")

# Notes:

- family = 0 means goodware | everything else is malware/ransomware
- can reduce dimensionality by summing API category counts, DLL category counts, suspicious capability flags, packing/obfuscation factors, ratios and summary stats

- ransomware behavior usually combines: file discovery, encryption, file rewrite/rename/delete, 
- persistence or privilege use, optional C2 communication, occasional backup disablement

In [5]:
# column grouping

# columns that are related to file API calls
file_api_cols = [
    "createfilea", "readfile", "writefile", "findfirstfilea",
    "findnextfilea", "copyfilea", "movefilea", "deletefilea",
    "setfileattributesa", "setendoffile"
]

# columns that are related to crypto API calls
crypto_api_cols = [
    "bcryptencrypt", "bcryptopenalgorithmprovider",
    "bcryptgeneratesymmetrickey", "cryptacquirecontexta",
    "cryptcreatehash", "cryptencrypt", "cryptgenkey",
    "cryptimportkey", "crypthashdata"
]

# columns that are related to registry API calls
registry_api_cols = [
    "regopenkeya", "regsetkeyvaluew", "regdeletekeya",
    "regdeletevaluea", "regnotifychangekeyvalue"
]

# columns that are related to network API calls
network_api_cols = [
    "internetopena", "internetopenurla", "internetreadfile",
    "httpopenrequesta", "httpsendrequesta", "winhttpopen",
    "winhttpconnect", "socket"
]

### Define functions

In [6]:
def present_data(df, cols):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.Series(0, index=df.index)
    return df[cols].sum(axis=1)

In [7]:
def random_forest_model(X_TRAIN, Y_TRAIN):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_TRAIN, Y_TRAIN)
    return model

In [16]:
def evaluate_model(model, X_TEST, Y_TEST):
    y_pred = model.predict(X_TEST)
    print ("Printing classification report --------------------------------------------")
    print(classification_report(Y_TEST, y_pred))
    print ("Printing confusion matrix --------------------------------------------")
    print(confusion_matrix(Y_TEST, y_pred))
    print ("Printing accuracy score --------------------------------------------")
    print(accuracy_score(Y_TEST, y_pred))

# Feature Extraction

In [ ]:
print ("extracting training features here --------------------------------------------")
    
X_TRAIN["num_file_apis"] = present_data(X_TRAIN, file_api_cols)
X_TRAIN["num_crypto_apis"] = present_data(X_TRAIN, crypto_api_cols)
X_TRAIN["num_registry_apis"] = present_data(X_TRAIN, registry_api_cols)
X_TRAIN["num_network_apis"] = present_data(X_TRAIN, network_api_cols)

X_TRAIN["has_crypto"] = (X_TRAIN["num_crypto_apis"] > 0).astype(int)
X_TRAIN["has_file_access"] = (X_TRAIN["num_file_apis"] > 0).astype(int)
X_TRAIN["has_network"] = (X_TRAIN["num_network_apis"] > 0).astype(int)

if "SizeOfImage" in X_TRAIN.columns and "SizeOfCode" in X_TRAIN.columns:
    X_TRAIN["code_to_image_ratio"] = X_TRAIN["SizeOfCode"] / X_TRAIN["SizeOfImage"].replace(0,1)

if "IMPORT_size" in X_TRAIN.columns and "SizeOfImage" in X_TRAIN.columns:
    X_TRAIN["import_to_image_ratio"] = X_TRAIN["IMPORT_size"] / X_TRAIN["SizeOfImage"].replace(0,1)
    
packer_columns = [c for c in X_TRAIN.columns if c.lower() in {"upx0", "upx1", "upx2", ".aspack", ".mpress1", ".mpress2"}]
if packer_columns:
    X_TRAIN["has_packer"] = (X_TRAIN[packer_columns].sum(axis=1) > 0).astype(int)
else:
    X_TRAIN["has_packer"] = 0
    
print("finished training extracting features ------------------------------------------")

extracting features here --------------------------------------------
finished extracting features ------------------------------------------


#### The same changes applied to the training dataset must be applied to the testing dataset.

In [14]:
print ("extracting testing features here --------------------------------------------")
    
X_TEST["num_file_apis"] = present_data(X_TEST, file_api_cols)
X_TEST["num_crypto_apis"] = present_data(X_TEST, crypto_api_cols)
X_TEST["num_registry_apis"] = present_data(X_TEST, registry_api_cols)
X_TEST["num_network_apis"] = present_data(X_TEST, network_api_cols)

X_TEST["has_crypto"] = (X_TEST["num_crypto_apis"] > 0).astype(int)
X_TEST["has_file_access"] = (X_TEST["num_file_apis"] > 0).astype(int)
X_TEST["has_network"] = (X_TEST["num_network_apis"] > 0).astype(int)

if "SizeOfImage" in X_TEST.columns and "SizeOfCode" in X_TEST.columns:
    X_TEST["code_to_image_ratio"] = X_TEST["SizeOfCode"] / X_TEST["SizeOfImage"].replace(0,1)

if "IMPORT_size" in X_TEST.columns and "SizeOfImage" in X_TEST.columns:
    X_TEST["import_to_image_ratio"] = X_TEST["IMPORT_size"] / X_TEST["SizeOfImage"].replace(0,1)
    
packer_columns = [c for c in X_TEST.columns if c.lower() in {"upx0", "upx1", "upx2", ".aspack", ".mpress1", ".mpress2"}]
if packer_columns:
    X_TEST["has_packer"] = (X_TEST[packer_columns].sum(axis=1) > 0).astype(int)
else:
    X_TEST["has_packer"] = 0
    
print("finished testing extracting features ------------------------------------------")

extracting testing features here --------------------------------------------
finished testing extracting features ------------------------------------------


# Training Random Forest Model

In [11]:
print("training random forest model ------------------------------------------")
rf_model = random_forest_model(X_TRAIN, Y_TRAIN.values.ravel())

training random forest model ------------------------------------------


# Evaluating Random Forest Model

In [17]:
print("evaluating random forest model ------------------------------------------")
evaluate_model(rf_model, X_TEST, Y_TEST)

print("finished --------------------------------------------------------------")

evaluating random forest model ------------------------------------------
Printing classification report --------------------------------------------
              precision    recall  f1-score   support

           0       0.66      0.97      0.79       133
           1       0.99      0.83      0.90       385

    accuracy                           0.86       518
   macro avg       0.82      0.90      0.84       518
weighted avg       0.90      0.86      0.87       518

Printing confusion matrix --------------------------------------------
[[129   4]
 [ 66 319]]
Printing accuracy score --------------------------------------------
0.8648648648648649
finished --------------------------------------------------------------
